In [ ]:
prefix_path = '../..'

import sys
sys.path.append(prefix_path)

import logging
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

from detection_baselines.detection_halp import (
    load_halp_dataset, halp_collate_fn,
    HalpLinearProbe, train_halp_probe, eval_halp_probe
)
from egh_vlm.utils import Qwen3ModelBundle, load_phd_dataset, split_stratified

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3_vl_2b.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/halp_phd_qwen3_layer24.pt'
DETECTOR_PATH = f'{prefix_path}/data/phd/detectors/halp_detector_phd_qwen3_layer24.pt'
OUTPUT_PATH = f'{prefix_path}/data/phd/evaluations/evaluation_halp_phd_qwen3_layer24.json'
LOGGING_PATH = f'{prefix_path}/data/logs/log_halp_phd_qwen3_layer24.txt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
TRAIN_RATIO = 0.7

#### Load Dataset

In [ ]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH,
)
print(f'Length of dataset: {len(dataset)}')

features = load_halp_dataset(FEATURES_PATH)
print(f'Length of features: {len(features)}')

hidden_size = features.hiddens[0].size(-1) if len(features) > 0 else 0
print(f'Hidden size: {hidden_size}')

In [ ]:
train_dataset, test_dataset = split_stratified(features, train_ratio=TRAIN_RATIO, random_state=42)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    collate_fn=halp_collate_fn,
    shuffle=True,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=halp_collate_fn,
    shuffle=True,
)

#### Experiment

In [ ]:
def train(
    train_dataloader: DataLoader,
    test_dataloader: DataLoader,
    hidden_size: int,
    logging_path: str,
    device: torch.device,
    train_ratio: float = 0.7,
    epochs_count: int = 30,
    lr: float = 1e-4,
):
    logging.basicConfig(filename=logging_path, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    logging.info(f'Starting training with hidden_size={hidden_size}, train_ratio={train_ratio}, epochs_count={epochs_count}, lr={lr}')

    result = {
        'train_ratio': train_ratio,
        'epochs_count': epochs_count,
        'lr': lr,
        'acc':    {'value': 0.0, 'epoch': -1},
        'f1':     {'value': 0.0, 'epoch': -1},
        'pr_auc': {'value': 0.0, 'epoch': -1},
    }

    probe = HalpLinearProbe(input_dim=hidden_size, device=device)
    loss_fn = nn.BCELoss()
    optim = torch.optim.Adam(probe.parameters(), lr=lr)

    for i in range(epochs_count):
        loss = train_halp_probe(probe, loss_fn, optim, train_dataloader)
        metrics = eval_halp_probe(probe, test_dataloader)
        acc, f1, pr_auc = metrics['acc'], metrics['f1'], metrics['pr_auc']

        logging.info(f'Epoch {i+1}/{epochs_count}, Loss: {loss:.4f}')
        logging.info(f'Epoch {i+1}/{epochs_count}, Accuracy: {acc:.4f}, F1: {f1:.4f}, PR-AUC: {pr_auc:.4f}\n')

        if acc > result['acc']['value']:
            result['acc'] = {'value': acc, 'epoch': i + 1}
        if f1 > result['f1']['value']:
            result['f1'] = {'value': f1, 'epoch': i + 1}
            os.makedirs(os.path.dirname(DETECTOR_PATH), exist_ok=True)
            torch.save(probe.state_dict(), DETECTOR_PATH)
        if pr_auc > result['pr_auc']['value']:
            result['pr_auc'] = {'value': pr_auc, 'epoch': i + 1}

    logging.info(f'Best Accuracy: {result["acc"]["value"]:.4f} at epoch {result["acc"]["epoch"]}')
    logging.info(f'Best F1: {result["f1"]["value"]:.4f} at epoch {result["f1"]["epoch"]}')
    logging.info(f'Best PR-AUC: {result["pr_auc"]["value"]:.4f} at epoch {result["pr_auc"]["epoch"]}')

    logger = logging.getLogger()
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)

    return probe, result

In [ ]:
os.makedirs(os.path.dirname(LOGGING_PATH), exist_ok=True)
probe, result = train(
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    hidden_size=hidden_size,
    logging_path=LOGGING_PATH,
    device=DEVICE,
    train_ratio=TRAIN_RATIO,
)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w') as f:
    json.dump(result, f, indent=2)

print(f'Best F1:     {result["f1"]["value"]:.4f} (epoch {result["f1"]["epoch"]})')
print(f'Best PR-AUC: {result["pr_auc"]["value"]:.4f} (epoch {result["pr_auc"]["epoch"]})')
print(f'Best Acc:    {result["acc"]["value"]:.4f} (epoch {result["acc"]["epoch"]})')